Ps 05
=====

**Author:** Erik An

**Date:** <2026-04-27 Mon>



## Problem 44



### Task



> Searching an unsorted array 1 This problem examines an algorithm for searching for a value $x$ in an unsorted array $A$ consisting of $n$ elements called `random_search`. Consider the following randomized strategy: pick a random index $i$ into $A$. If $A[i] = x$, then terminate; otherwise, continue the search by picking a new random index into $A$. Continue picking random indices into $A$ until you find an index $j$ such that $A[j] = x$ or until every element of $A$ has been checked. This strategy may examine a given element more than once, because it picks from the whole set of indices each time. In the following we assume that all possible permutations of the input array are equally likely.
> 
> (a) Implement a procedure `random_search` to realise the strategy above. Be sure that your algorithm terminates when all indices into $A$ have been picked.
> 
> (b) Suppose that there is exactly one index $i$ such that $A[i] = x$. What is the expected number of indices into $A$ that must be picked before $x$ is found and `random_search` terminates?
> 
> (c) Suppose now that there are $k > 1$ indices $i$ such that $A[i] = x$. What is the expected number of indices into $A$ that must be picked before $x$ is found and random search terminates? Your answer should be a function of $n$ and $k$.
> 
> (d) Suppose that there are no indices $i$ such that $A[i] = x$. What is the expected number of indices into $A$ that must be picked before all elements of $A$ have been checked and random search terminates?\*\* Solution



#### (a)



In [1]:
using Random

let
    function random_search(A, x)
        seen = falses(length(A))
        count = 0

        while !all(seen)
            count += 1
            i = rand(eachindex(A))
            A[i] == x && return (i, count)
            seen[i] = true
        end

        return (nothing, count)
    end

    arr = [1, 2, 3, 4, 5, 7, 7, 8, 9, 0]
    idx, _ = random_search(arr, 7)
    println("found at index $idx")
end

found at index 6

#### (b)



In [1]:
using Random

let
    function random_search(A, x)
        seen = falses(length(A))
        count = 0

        while !all(seen)
            count += 1
            i = rand(eachindex(A))
            A[i] == x && return (i, count)
            seen[i] = true
        end

        return (nothing, count)
    end

    mean_picks(A, x; trials=1_000_000) = sum(random_search(A, x)[2] for _ in 1:trials) / trials

    arr = [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
    println("E(T) ≈ $(mean_picks(arr, 7))")
end

E(T) ≈ 9.993828

Let $n = |A|$. Each pick is i.i.d. Bernoulli$(1/n)$: independence from sampling with replacement, identical distribution from uniform index choice. Let $T$ be the number of picks until the first success.

When $x \in A$, the unique successful index lies among the $n$ indices, so the algorithm returns before the `seen` guard fires; the bounded $T$ equals the unbounded geometric $T$ almost surely.

\begin{align*}
P(T = k) &= (1 - p)^{k-1} p, \quad k = 1, 2, \dots \\
E(T) &= \sum_{k=1}^{\infty} k \, (1 - p)^{k-1} p \\
     &= p \sum_{k=1}^{\infty} k (1 - p)^{k-1} \\
     &= p \cdot \frac{1}{(1 - (1 - p))^2} \quad \left( \sum_{k=1}^{\infty} k x^{k-1} = \frac{1}{(1-x)^2}, \; |x| < 1 \right) \\
     &= \frac{1}{p} = n.
\end{align*}

Sanity check: $n = 10$, predicted $E(T) = 10$, empirical $9.993$.



#### (c)



In [1]:
using Random

let
    function random_search(A, x)
        seen = falses(length(A))
        count = 0

        while !all(seen)
            count += 1
            i = rand(eachindex(A))
            A[i] == x && return (i, count)
            seen[i] = true
        end

        return (nothing, count)
    end

    mean_picks(A, x; trials=1_000_000) = sum(random_search(A, x)[2] for _ in 1:trials) / trials

    arr = [1, 2, 3, 4, 5, 7, 7, 8, 9, 0]
    println("E(T) ≈ $(mean_picks(arr, 7))")
end

E(T) ≈ 4.998466

By the same argument as (b): each pick is i.i.d. Bernoulli$(p)$, but now $p = k/n$, since $k$ of the $n$ indices yield $A[i] = x$. The `seen` guard does not fire (at least one successful index exists, so the loop returns first), so $T$ is geometric. Hence

$$E(T) = \frac{1}{p} = \frac{n}{k}.$$

Sanity check: $n = 10$, $k = 2$, predicted $E(T) = 5$, empirical $4.998$.



#### (d)



In [1]:
using Random

let
    function random_search(A, x)
        seen = falses(length(A))
        count = 0

        while !all(seen)
            count += 1
            i = rand(eachindex(A))
            A[i] == x && return (i, count)
            seen[i] = true
        end

        return (nothing, count)
    end

    mean_picks(A, x; trials=1_000_000) = sum(random_search(A, x)[2] for _ in 1:trials) / trials

    arr = [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
    println("E(T) ≈ $(mean_picks(arr, 10))")
end

E(T) ≈ 29.294806

Define $T_i$ be a number of picks made while you have exactly $i-1$ distinct indices seen, until the new $i$ th element was picked. Hence, $T = T_1 + T_2 + \ldots + T_n$.

Each $T_i$ is geometric, at the moment $i - 1$ was picked, the probability of picking new is $p_i = \frac{(n-i+1)}{n}$.

\begin{align*}
E(T) &= E\!\left(\sum_{i=1}^{n} T_i\right) \\
     &= \sum_{i=1}^{n} E(T_i) \qquad \text{(linearity of expectation)} \\
     &= \sum_{i=1}^{n} \frac{1}{p_i} \qquad \text{(by (b), since } T_i \sim \mathrm{Geometric}(p_i)\text{)} \\
     &= \sum_{i=1}^{n} \frac{n}{n - i + 1} \\
     &= n \sum_{j=1}^{n} \frac{1}{j} \qquad (j = n - i + 1) \\
     &= n H_n.
\end{align*}

Sanity check: $n = 10$, predicted $E(T) \approx 29.29$, empirical $29.29$.



## Problem 45



### Task



> Searching an unsorted array 2 This problem examines the search algorithm `deterministic_search`. This algorithm searches $A$ for $x$ in order, considering $A[1], A[2], \ldots A[n]$, until either it finds $A[i] = x$ or it reaches the end of the array. In the following we again assume that all possible permutations of the input array are equally likely.
> 
> (a) Implement a procedure `deterministic_search` to realise the strategy above.
> 
> (b) Suppose that there is exactly one index $i$ such that $A[i] = x$. What is the average-case running time of `deterministic_search`? What is its worst-case running time?
> 
> (c) Suppose that there are $k > 1$ indices $i$ such that $A[i] = x$. What is the average-case running time of `deterministic_search`? What is the worst-case running time of `deterministic_search`? Your answer should be a function of $n$ and $k$.
> 
> (d) Suppose that there are no indices $i$ such that $A[i] = x$. What is the average-case running time of `deterministic_search`? What is its worst-case running time?\*\* Solution



#### (a)



In [1]:
using Random

let
    function deterministic_search(A, x)
        for i in eachindex(A)
            A[i] == x && return i
        end
        return nothing
    end

    arr = [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
    idx = deterministic_search(arr, 7)
    println("found at index $idx")
end

found at index 7

#### (b)



Assume uniform distribution of all possible permutations of an array, with $p_i = \frac{1}{n}$, where $n = |A|$.

By definition of the algorithm, it does $i$ comparisons to reach position $i$. Hence,

\begin{align*}
T_i &= i \\
T_{\text{avg}} &= \sum_{i=1}^{n} p_i \cdot T_i \\
&= \sum_{i=1}^{n} \frac{i}{n} \\
&= \frac{1}{n} \cdot \sum_{i=1}^{n} i \\
&= \frac{1}{n} \cdot \frac{n(n+1)}{2} \\
&= \frac{n+1}{2} \\
\end{align*}

Worst case running time would be traversing through every element of the array, hence $\Theta(n)$.



#### (c)



In [1]:
using Random

let
    function deterministic_search(A, x)
        for i in eachindex(A)
            A[i] == x && return i
        end
        return nothing
    end

    mean_picks(A, x; trials=1_000_000) = sum(deterministic_search(shuffle(A), x) for _ in 1:trials) / trials

    arr = [1, 2, 3, 4, 5, 7, 7, 8, 9, 0]
    println("T_avg ≈ $(mean_picks(arr, 7))")
end

T_avg ≈ 3.667191

\begin{align*}
P(\text{first match at } j) &= \frac{\binom{n-j}{k-1}}{\binom{n}{k}} \\
&= \frac{\text{arrangements with first x at position j}}{\text{total arrangement of x's positions}} \\
T_{\text{avg}} &= \sum_{j = 1}^{n - k + 1} j \cdot P(\text{first match at } j) \\
&= \sum_{j = 1}^{n - k + 1} j \cdot \frac{\binom{n-j}{k-1}}{\binom{n}{k}} \\
\end{align*}

*Note, this expression can be simplified, I just do not want to do it.*

Worst case running time would be traversing through every non $x$ element of the array, hence $\Theta(n-k+1)$.

\begin{align*}
T_{\text{avg}} &= \sum_{j = 1}^{10 - 2 + 1} j \cdot \frac{\binom{10-j}{2-1}}{\binom{10}{2}} \\
&= \frac{11}{3} \\
&\approx 3.666 \\
\end{align*}

Sanity check: $n = 10, k = 2$, predicted $T_{\text{avg}} \approx 3.666$, empirical $3.667$.



#### (d)



In [1]:
using Random

let
    function deterministic_search(A, x)
        count = 0

        for i in eachindex(A)
            count += 1
            if A[i] == x
                break
            end
        end
        return count
    end

    mean_picks(A, x; trials=1_000_000) = sum(deterministic_search(shuffle(A), x) for _ in 1:trials) / trials

    arr = [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
    println("T_avg ≈ $(mean_picks(arr, 10))")
end

T_avg ≈ 10.0

As there is no $i : A[i] = x$, the algorithm would traverse whole array, resulting in $\Theta(n)$, where $n = |A|$.

Sanity check: $n = 10, k = 0$, predicted $P(\text{first match at } j) = n = 10$, empirical $10.0$.



## Problem 46



### Task



> Searching an unsorted array 3 This problem examines the search algorithm `scramble_search`. It first randomly permutes the input array and then runs the algorithm `deterministic_search` on the resulting permuted array. In the following we also assume that all possible permutations of the input array are equally likely.
> 
> (a) Implement a procedure `scramble_search` to realise the strategy above. Use your implementation of `deterministic_search` from Exercise 45.
> 
> (b) Suppose that there is exactly one index $i$ such that $A[i] = x$. What is the expected number of indices into $A$ that must be picked before $x$ is found and `scramble_search` terminates?
> 
> (c) Suppose now that there are $k > 1$ indices $i$ such that $A[i] = x$. What is the expected number of indices into $A$ that must be picked before $x$ is found and `scramble_search` terminates? Your answer should be a function of $n$ and $k$.
> 
> (d) Suppose that there are no indices $i$ such that $A[i] = x$. What is the expected number of indices into $A$ that must be picked before all elements of $A$ have been checked and `scramble_search` terminates?
> 
> Which of the three searching algorithms `random_search`, `deterministic_search`, or `scramble_search` would you use and why?\*\* Solution



#### (a)



In [1]:
using Random

let
    function deterministic_search(A, x)
        for i in eachindex(A)
            if A[i] == x
                return i
            end
        end
        return nothing
    end

    function scramble_search(A, x)
        return deterministic_search(shuffle(A), x)
    end

    arr = [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
    idx = scramble_search(arr, 7)
    println("found at index $idx")
end

found at index 9

#### (b)



Let $J$ be the index of $x$ after shuffling. Since all permutations are equiprobable, $J \equiv \mathrm{Uniform}\{1, \ldots, n \}$, and $T = J$.

$$E(T) = \sum_{j=1}^n j \cdot \frac{1}{n} = \frac{n+1}{2}$$



#### (c)



In [1]:
using Random

let
    function deterministic_search(A, x)
        for i in eachindex(A)
            A[i] == x && return i
        end
        return nothing
    end

    scramble_search(A, x) = deterministic_search(shuffle(A), x)

    mean_picks(A, x; trials=1_000_000) = sum(scramble_search(A, x) for _ in 1:trials) / trials

    arr = [1, 2, 3, 4, 5, 7, 7, 8, 9, 0]
    println("E(T) ≈ $(mean_picks(arr, 7))")
end

E(T) ≈ 3.669679

Let $J$ be the position of the first match after shuffling. With $k$ matches placed uniformly at random among $n$ positions:

\begin{align*}
P(J = j) &= \frac{\binom{n-j}{k-1}}{\binom{n}{k}}, \quad 1 \le j \le n - k + 1 \\
E(J) &= \sum_{j=1}^{n-k+1} j \cdot \frac{\binom{n-j}{k-1}}{\binom{n}{k}}
\end{align*}

Substituting $i = n - j$ and applying the hockey-stick identity twice, using:

$$i\binom{i}{k-1} = k\binom{i}{k} + (k-1)\binom{i}{k-1}$$

\begin{align*}
\sum_{j=1}^{n-k+1} j \binom{n-j}{k-1}
  &= n\binom{n}{k} - k\binom{n}{k+1} - (k-1)\binom{n}{k} \\
  &= (n-k+1)\binom{n}{k} - k\binom{n}{k+1} \\
  &= \frac{n+1}{k+1}\binom{n}{k}
\end{align*}

Hence

$$E(T) = \frac{n+1}{k+1}$$

Sanity: $n=10, k=2 \Rightarrow \frac{11}{3} \approx 3.667$, matching the shuffle simulation from Problem 44(c).



#### (d)



In [1]:
using Random

let
    function deterministic_search(A, x)
        for i in eachindex(A)
            A[i] == x && return i
        end
        return length(A)
    end

    scramble_search(A, x) = deterministic_search(shuffle(A), x)

    mean_picks(A, x; trials=1_000_000) = sum(scramble_search(A, x) for _ in 1:trials) / trials

    arr = [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
    println("E(T) ≈ $(mean_picks(arr, 10))")
end

E(T) ≈ 10.0

With no $i$ such that $A[i] = x$, every permutation forces the algorithm to examine all $n$ elements before terminating. Hence $T = n$ deterministically and

$$E(T) = n$$

Sanity: $n = 10 \Rightarrow E(T) = 10$, empirical $10.0$ (zero variance, as expected).



## Problem 47



### Task



> Scheduling In the Linux system all processes get a base priority value between 0 and 139. The process scheduler computes a real priority value from the process&rsquo;s base priority value. Lower priority values are better than higher priority values, i.e. from two processes the one with higher priority is the one with lower real priority value. Assume that the process scheduler always picks the next process among those with lowest real priority value. Assume further that every process has in addition to its two priority values the following additional information of the specified type: process id (integer), process name (string), user id (integer), group id (integer), registers (array(integer) of size 20), process flags (array(boolean) of size 20), floating point registers (array(double) of size 16).
> 
> Implement a heap data structure that can be used for the priority queue as described above.\*\* Solution



In [1]:
let
    mutable struct Node
        base_priority::UInt8
        real_priority::UInt8
        process_id::Int64
        process_name::String
        user_id::Int64
        group_id::Int64
        registers::NTuple{20, Int64}
        process_flags::NTuple{20, Bool}
        f_registers::NTuple{16, Float64}
    end

    struct MinHeap
        data::Vector{Node}
        MinHeap() = new(Node[])
    end

    #------------------------------------------

    parent(i) = i ÷ 2
    left(i)   = 2i
    right(i)  = 2i + 1

    #------------------------------------------

    function Base.push!(h::MinHeap, n::Node)
        d = h.data
        push!(d, n)
        i = length(d)
        while i > 1 && d[parent(i)].real_priority > d[i].real_priority
            d[parent(i)], d[i] = d[i], d[parent(i)]
            i = parent(i)
        end
        return h
    end

    function Base.pop!(h::MinHeap)
        d = h.data
        isempty(d) && return nothing
        result = d[1]
        d[1] = d[end]
        pop!(d)
        i = 1
        while left(i) <= length(d)
            j = left(i)
            if right(i) <= length(d) && d[right(i)].real_priority < d[j].real_priority
                j = right(i)
            end
            d[i].real_priority <= d[j].real_priority && break
            d[i], d[j] = d[j], d[i]
            i = j
        end
        return result
    end

    Base.peek(h::MinHeap) = h.data[1]

    #------------------------------------------

    make_node(p) = Node(0x00, UInt8(p), 0, "", 0, 0, ntuple(_->0,20), ntuple(_->false,20), ntuple(_->0.0,16))

    ps = [50, 10, 90, 30, 70, 20, 40, 80, 60, 5, 100, 1]
    h = MinHeap()
    foreach(p -> push!(h, make_node(p)), ps)
    @assert [Int(pop!(h).real_priority) for _ in ps] == sort(ps)
    println("ok")

end

ok

## Problem 48



### Task



> Implement a priority queue data structure for the process scheduler using the implemented heap of Exercise 47 and handles.\*\* Solution



In [1]:
let
    mutable struct Handle
        idx::Int
    end

    mutable struct Node
        base_priority::UInt8
        real_priority::UInt8
        process_id::Int64
        process_name::String
        user_id::Int64
        group_id::Int64
        registers::NTuple{20, Int64}
        process_flags::NTuple{20, Bool}
        f_registers::NTuple{16, Float64}
    end

    struct MinHeap
        data::Vector{Node}
        handles::Vector{Handle}
        MinHeap() = new(Node[], Handle[])
    end

    #------------------------------------------

    parent(i) = i ÷ 2
    left(i)   = 2i
    right(i)  = 2i + 1

    #------------------------------------------

    function swap!(h::MinHeap, i::Int, j::Int)
        h.data[i], h.data[j] = h.data[j], h.data[i]
        h.handles[i], h.handles[j] = h.handles[j], h.handles[i]
        h.handles[i].idx = i
        h.handles[j].idx = j
    end

    function siftup!(h::MinHeap, i::Int)
        d = h.data
        while i > 1 && d[parent(i)].real_priority > d[i].real_priority
            swap!(h, i, parent(i))
            i = parent(i)
        end
    end

    function siftdown!(h::MinHeap, i::Int)
        d = h.data
        while left(i) <= length(d)
            j = left(i)
            if right(i) <= length(d) && d[right(i)].real_priority < d[j].real_priority
                j = right(i)
            end
            d[i].real_priority <= d[j].real_priority && break
            swap!(h, i, j)
            i = j
        end
    end

    function Base.push!(h::MinHeap, n::Node)
        push!(h.data, n)
        i = length(h.data)
        handle = Handle(i)
        push!(h.handles, handle)
        siftup!(h, i)
        return handle
    end

    function Base.pop!(h::MinHeap)
       isempty(h.data) && return nothing
       result = h.data[1]
       popped_handle = h.handles[1]
       swap!(h, 1, length(h.data))
       pop!(h.data); pop!(h.handles)
       popped_handle.idx = 0
       siftdown!(h, 1)
       return result
   end

    Base.peek(h::MinHeap) = h.data[1]

    function decrease_key(h::MinHeap, handle::Handle, rp)
        handle.idx == 0 && return
        @assert rp <= h.data[handle.idx].real_priority
        h.data[handle.idx].real_priority = UInt8(rp)
        siftup!(h, handle.idx)
    end

    function increase_key(h::MinHeap, handle::Handle, rp)
        handle.idx == 0 && return
        @assert rp >= h.data[handle.idx].real_priority
        h.data[handle.idx].real_priority = UInt8(rp)
        siftdown!(h, handle.idx)
    end

    #------------------------------------------

    make_node(p) = Node(0x00, UInt8(p), 0, "", 0, 0, ntuple(_->0,20), ntuple(_->false,20), ntuple(_->0.0,16))

    ps = [50, 10, 90, 30, 70, 20, 40, 80, 60, 5, 100, 1]
    h = MinHeap()
    foreach(p -> push!(h, make_node(p)), ps)
    @assert [Int(pop!(h).real_priority) for _ in ps] == sort(ps)
    h2 = MinHeap()
    hs = [push!(h2, make_node(p)) for p in [50, 80, 30]]
    decrease_key(h2, hs[2], 5)
    @assert pop!(h2).real_priority == 5
    println("ok")
end

ok

## Problem 49



### Task



> Implement a simple scheduler that performs a loop, such that every iteration performs the following algorithm:
> 
> (a) Pick and remove the highest priority process $P$ from the priority queue.
> 
> (b) Decrease the real priority value of every process still in the queue by 1.
> 
> (c) With a 10% chance discard the process $P$ (it has terminated), with a 90% chance re-insert the process $P$ into the queue with a new real priority value of 10 times its base priority value plus 1.
> 
> (d) With a 10% chance randomly create a new process with random base priority in the range $[1, 138]$ and put it in the queue with real priority value equal to its base priority value.
> 
> Then populate the priority queue with a process called &ldquo;kernel&rdquo; with process id 0, user id 0, group id 0, base priority value 1, the rest random, a process called &ldquo;idle&rdquo; with process id 1, user id 0, group id 0, base priority value 139, the rest random, and three other completely random processes. Then perform 1000 iterations of your process scheduler and print the process that was chosen in every iteration.\*\* Solution



In [1]:
using Random

let
    mutable struct Handle
        idx::Int
    end

    mutable struct Node
        base_priority::Int
        real_priority::Int
        process_id::Int
        process_name::String
        user_id::Int
        group_id::Int
        registers::NTuple{20, Int64}
        process_flags::NTuple{20, Bool}
        f_registers::NTuple{16, Float64}
    end

    struct MinHeap
        data::Vector{Node}
        handles::Vector{Handle}
        MinHeap() = new(Node[], Handle[])
    end

    #------------------------------------------

    parent(i) = i ÷ 2
    left(i)   = 2i
    right(i)  = 2i + 1

    #------------------------------------------

    function swap!(h::MinHeap, i::Int, j::Int)
        h.data[i], h.data[j] = h.data[j], h.data[i]
        h.handles[i], h.handles[j] = h.handles[j], h.handles[i]
        h.handles[i].idx = i
        h.handles[j].idx = j
    end

    function siftup!(h::MinHeap, i::Int)
        d = h.data
        while i > 1 && d[parent(i)].real_priority > d[i].real_priority
            swap!(h, i, parent(i))
            i = parent(i)
        end
    end

    function siftdown!(h::MinHeap, i::Int)
        d = h.data
        while left(i) <= length(d)
            j = left(i)
            if right(i) <= length(d) && d[right(i)].real_priority < d[j].real_priority
                j = right(i)
            end
            d[i].real_priority <= d[j].real_priority && break
            swap!(h, i, j)
            i = j
        end
    end

    function Base.push!(h::MinHeap, n::Node)
        push!(h.data, n)
        i = length(h.data)
        handle = Handle(i)
        push!(h.handles, handle)
        siftup!(h, i)
        return handle
    end

    function Base.pop!(h::MinHeap)
       isempty(h.data) && return nothing
       result = h.data[1]
       popped_handle = h.handles[1]
       swap!(h, 1, length(h.data))
       pop!(h.data); pop!(h.handles)
       popped_handle.idx = 0
       siftdown!(h, 1)
       return result
   end

    Base.peek(h::MinHeap) = h.data[1]

    function decrease_key(h::MinHeap, handle::Handle, rp)
        handle.idx == 0 && return
        @assert rp <= h.data[handle.idx].real_priority
        h.data[handle.idx].real_priority = Int(rp)
        siftup!(h, handle.idx)
    end

    function increase_key(h::MinHeap, handle::Handle, rp)
        handle.idx == 0 && return
        @assert rp >= h.data[handle.idx].real_priority
        h.data[handle.idx].real_priority = Int(rp)
        siftdown!(h, handle.idx)
    end

    #------------------------------------------

    function random_node(; pid=rand(Int), uid=rand(Int), gid=rand(Int), base=rand(1:138), name=randstring(8))
        Node(
            base,
            base,
            pid, name, uid, gid,
            ntuple(_ -> rand(Int),    20),
            ntuple(_ -> rand(Bool),   20),
            ntuple(_ -> rand(),       16),
        )
    end

    function step!(h::MinHeap)
        isempty(h.data) && return nothing
        hp = pop!(h)
        for n in h.data
            n.real_priority -= 1
        end
        if rand() < 0.9
            hp.real_priority = hp.base_priority * 10 + 1
            push!(h, hp)
        end
        if rand() < 0.1
            push!(h, random_node())
        end
        return hp
    end

    function scheduler(h::MinHeap, n::Int=1000)
        for _ in 1:n
            p = step!(h)
            (p === nothing) || println(p.process_name, " (pid=", p.process_id, ")")
        end
    end

    h = MinHeap()
    push!(h, random_node(pid=0, uid=0, gid=0, base=1,   name="kernel"))
    push!(h, random_node(pid=1, uid=0, gid=0, base=139, name="idle"))
    for _ in 1:3
        push!(h, random_node())
    end
    scheduler(h, 1000)
end

#+begin_example
kernel (pid=0)
kernel (pid=0)
kernel (pid=0)
kernel (pid=0)
kernel (pid=0)
kernel (pid=0)
kernel (pid=0)
kernel (pid=0)
kernel (pid=0)
kernel (pid=0)
kernel (pid=0)
kernel (pid=0)
sO81yWfI (pid=6098878896715955361)
00u2vhqy (pid=-4085129693727430007)
UeYdIKY0 (pid=-7807681412908936421)
2PUlyQ1x (pid=65165999772993361)
YUDhTGqa (pid=4004126849518487845)
idle (pid=1)
lGkcuy6l (pid=-4497509044890938028)
BZ2PbEL9 (pid=2900543848609925147)
sO81yWfI (pid=6098878896715955361)
sO81yWfI (pid=6098878896715955361)
sO81yWfI (pid=6098878896715955361)
sO81yWfI (pid=6098878896715955361)
sO81yWfI (pid=6098878896715955361)
sO81yWfI (pid=6098878896715955361)
00u2vhqy (pid=-4085129693727430007)
00u2vhqy (pid=-4085129693727430007)
00u2vhqy (pid=-4085129693727430007)
UeYdIKY0 (pid=-7807681412908936421)
UeYdIKY0 (pid=-7807681412908936421)
UeYdIKY0 (pid=-7807681412908936421)
UeYdIKY0 (pid=-7807681412908936421)
2PUlyQ1x (pid=65165999772993361)
2PUlyQ1x (pid=65165999772993361)
2PUlyQ1x (pid=651

## Problem 50



### Task



> When RANDOMIZED-QUICKSORT runs, how many calls are made to the random-number generator RANDOM in the worst case? How about in the best case? Give your answer in terms of $\Theta$ notation.\*\* Solution

RANDOMIZED-PARTITION calls RANDOM exactly once each time it runs, so counting RANDOM calls is the same as counting partition calls. Let $f(n)$ be the number of partition calls on an array of size $n$, with $f(0) = f(1) = 0$ (no partition is called on an empty or singleton subarray). If a partition produces sides of sizes $k$ and $n - 1 - k$, then

$$
f(n) = 1 + f(k) + f(n - 1 - k).
$$

To bound $f(n)$ I look at the recursion tree. Let $I = f(n)$ be the number of internal nodes (partition calls) and $L = I + 1$ the number of leaves (a property of any binary tree where every internal node has two children). Each partition fixes exactly one element as its pivot, so out of the $n$ original elements, $n - I$ of them end up alone in size-1 leaves. The rest of the leaves are size-0:

$$
L_1 = n - I, \qquad L_0 = (I + 1) - L_1 = 2I + 1 - n.
$$

Since $L_0 \ge 0$, we get $I \ge \lceil (n-1)/2 \rceil$. So no matter what splits happen, $f(n) = \Omega(n)$.

-   Worst case
    
    This is when partitions are as unbalanced as possible: the pivot is always the smallest or largest element, so one side has size $0$ and the other has size $n - 1$. Then
    
    $$
        f(n) = 1 + f(n - 1), \quad f(1) = 0 \implies f(n) = n - 1 = \Theta(n).
        $$

-   Best case
    
    To minimize $f(n)$ I want to minimize the number of partition calls, which by the bound above means making $L_0$ as small as possible. Splitting into sizes $1$ and $n - 2$ at every step gives $L_0 = 0$ (when $n$ is odd) and saturates the bound:
    
    $$
        f(n) = 1 + f(1) + f(n - 2) = 1 + f(n - 2) = \lceil (n-1)/2 \rceil = \Theta(n).
        $$
    
    Note: in CLRS the &ldquo;best case&rdquo; usually means the median split, because that is what minimizes the **running time**. For just the RANDOM count, $(1, n-2)$ splits are actually fewer calls than median splits, but both are $\Theta(n)$, so the asymptotic answer is the same either way.

So both the best and worst case make $\Theta(n)$ calls to RANDOM. This is different from the number of comparisons, which is $\Theta(n \log n)$ in the best case and $\Theta(n^2)$ in the worst case. The reason RANDOM is the same in both cases: every partition costs exactly one coin flip, and the lower bound $I \ge \lceil (n-1)/2 \rceil$ already forces $\Theta(n)$ partition calls regardless of how balanced the splits are.

I checked this with a small simulation in Julia:



In [1]:
using Random
let
    function randomized_partition!(A, lo, hi)
        p = rand(lo:hi)
        A[p], A[hi] = A[hi], A[p]
        pivot = A[hi]
        i = lo - 1
        @inbounds for j in lo:hi-1
            if A[j] ≤ pivot
                i += 1
                A[i], A[j] = A[j], A[i]
            end
        end
        A[i+1], A[hi] = A[hi], A[i+1]
        return i + 1
    end
    function randomized_quicksort!(A, lo=1, hi=length(A))
        if lo < hi
            q = randomized_partition!(A, lo, hi)
            randomized_quicksort!(A, lo, q - 1)
            randomized_quicksort!(A, q + 1, hi)
        end
        return A
    end
    randomized_quicksort(A) = randomized_quicksort!(copy(A))

    # Counting RANDOM calls in the three split shapes
    worst(n)        = n ≤ 1 ? 0 : 1 + worst(n - 1)
    best(n)         = n ≤ 1 ? 0 : cld(n - 1, 2)
    function median_split(n)
        n ≤ 1 && return 0
        k = (n - 1) ÷ 2
        1 + median_split(k) + median_split(n - 1 - k)
    end

    A = [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
    println(randomized_quicksort(A))
    for n in (10, 100, 1000)
        println("n=$n  worst=$(worst(n))  best=$(best(n))  median=$(median_split(n))")
    end
end

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
n=10  worst=9  best=5  median=6
n=100  worst=99  best=50  median=63
n=1000  worst=999  best=500  median=511

The numbers confirm that all three cases grow linearly in $n$.



## Problem 51



### Task



> Professors Howard, Fine, and Howard have proposed a deceptively simple sorting algorithm, named stooge sort in their honor.
> 
> 1:   procedure STOOGE-SORT(A,p,r)
> 2:     if $A[p] > A[r]$ then
> 3:       exchange $A[p]$ with $A[r]$
> 4:     end if
> 5:     if $p + 1 < r$ then
> 6:       $k = \lfloor (r - p + 1)/3 \rfloor$
> 7:       STOOGE-SORT(A, p, r - k) */ first two-thirds
> 8:       STOOGE-SORT(A, p + k, r) /* last two-thirds
> 9:       STOOGE-SORT(A, p, r - k) // first two-thirds again
> 10:    end if
> 11:  end procedure
> 
> (a) Argue that the call STOOGE-SORT(A, 1, n) correctly sorts the array $A[1 : n]$.
> 
> (b) Give a recurrence for the worst-case running time of STOOGE-SORT and a tight asymptotic ($\Theta$ notation) bound on the worst-case running time.
> 
> (c) Compare the worst-case running time of STOOGE-SORT with that of insertion sort, merge sort, heapsort, and quicksort. Do the professors deserve tenure?\*\* Solution



#### (a)



Induction on $n = r - p + 1$.

*Base.* $n \le 2$: lines 2–3 perform the only nontrivial work and produce a sorted pair (or singleton); the recursion guard $p + 1 < r$ blocks further calls.

*Step.* $n \ge 3$, so $k = \lfloor n/3 \rfloor \ge 1$ and each recursive call has size $n - k = \lceil 2n/3 \rceil < n$. Let $L = A[p..r-k]$ and $R = A[p+k..r]$, both of size $n - k$, with $|L \cap R| = n - 2k \ge 0$ since $k \le n/3$.

By IH all three recursive calls sort their subarrays. Trace:

1.  After call 1 ($L$ sorted): the $k$ largest entries of $L$ sit at positions $[r - 2k + 1,\ r - k]$. This range lies inside $R$, since $p + k \le r - 2k + 1 \iff 3k \le n$, which holds.

2.  The $k$ globally largest entries of $A[p..r]$ are either originally in $L$ — now in its top-$k$ tail $\subset R$ — or in $A[r-k+1..r] \subset R$. Either way, after call 1 they all sit inside $R$.

3.  After call 2 ($R$ sorted): the top $k$ of $R$ — equal to the top $k$ of $A$ by step 2 — occupy $A[r-k+1..r]$ in sorted order.

4.  After call 3 ($L$ sorted): $L$ holds the $n - k$ smallest entries of $A$ in sorted order. Concatenated with the sorted $A[r-k+1..r]$, $A[p..r]$ is sorted.



#### (b)



Outside recursion the work is $\Theta(1)$ (one comparison, optional swap). Three recursive calls on size $\lceil 2n/3 \rceil$:

$$T(n) = 3\, T\!\left(\left\lceil \tfrac{2n}{3} \right\rceil\right) + \Theta(1).$$

Master theorem with $a = 3$, $b = 3/2$: $\log_b a = \log_{3/2} 3 = \dfrac{\ln 3}{\ln 3 - \ln 2} \approx 2.7095$. Since $f(n) = \Theta(1) = O(n^{\log_b a - \varepsilon})$ for small $\varepsilon$, case 1 applies:

$$T(n) = \Theta\!\left(n^{\log_{3/2} 3}\right) \approx \Theta(n^{2.71}).$$



In [1]:
using Random

let
    ops = Ref(0)

    function stooge_sort!(A, p=1, r=length(A))
        ops[] += 1
        if A[p] > A[r]
            A[p], A[r] = A[r], A[p]
        end
        if p + 1 < r
            k = (r - p + 1) ÷ 3
            stooge_sort!(A, p, r - k)
            stooge_sort!(A, p + k, r)
            stooge_sort!(A, p, r - k)
        end
        return A
    end

    Random.seed!(0)
    for _ in 1:500
        n = rand(1:40)
        A = rand(-100:100, n)
        @assert stooge_sort!(copy(A)) == sort(A)
    end
    println("correctness ok")

    α = log(3) / log(1.5)
    for n in (50, 100, 500, 1000)
        ops[] = 0
        stooge_sort!(rand(n))
        slope = log(ops[]) / log(n)
        println("n=$n  ops=$(ops[])  log(ops)/log(n) = $(round(slope, digits=4))")
    end
    println("α = log_{3/2}(3) ≈ $(round(α, digits=4))")
end

correctness ok
n=50  ops=29524  log(ops)/log(n) = 2.6311
n=100  ops=265720  log(ops)/log(n) = 2.7122
n=500  ops=21523360  log(ops)/log(n) = 2.7169
n=1000  ops=64570081  log(ops)/log(n) = 2.6033
α = log_{3/2}(3) ≈ 2.7095

Sanity check: empirical log–log slopes oscillate around $\alpha \approx 2.71$; the wobble comes from floor/ceiling effects in $k = \lfloor n/3 \rfloor$, not from a violated bound.



#### (c)




| Algorithm|Worst case|
|---|---|
| Insertion sort|$\Theta(n^2)$|
| Merge sort|$\Theta(n \log n)$|
| Heapsort|$\Theta(n \log n)$|
| Quicksort|$\Theta(n^2)$|
| Stooge sort|$\Theta(n^{\log_{3/2} 3}) \approx \Theta(n^{2.71})$|

Stooge sort is asymptotically worse than every algorithm in the comparison, including the quadratic ones. Tenure denied.



## Errors/Typos



-   Problem 44: &ldquo;called random<sub>search</sub>].&rdquo; → stray &ldquo;]&rdquo; (likely orphaned bracket from &ldquo;[random search]&rdquo; or footnote marker)
-   Problem 44(c): &ldquo;ramdom search&rdquo; → &ldquo;random search&rdquo;
-   Problem 45: &ldquo;This problem examine the search algorithm&rdquo; → &ldquo;This problem examines the search algorithm&rdquo;
-   Problem 47: &ldquo;all processes gets a base priority value&rdquo; → &ldquo;all processes get a base priority value&rdquo;
-   Problem 51: `end if` on line 7 is misplaced — the three recursive calls (lines 8–10) should be inside the `if p + 1 < r` block, otherwise $k$ is undefined when $p + 1 \geq r$. The `end if` should be on line 11 (after the recursive calls), and the procedure&rsquo;s `end procedure` on line 12.

